# 💾 BƯỚC 0: TIỀN XỬ LÝ & LẬP CHỈ MỤC VĨNH VIỄN (OFFLINE INDEXING)
> **Mục tiêu**: Đọc toàn bộ tài liệu PDF, thực hiện **Small-to-Big (Parent-Child) Chunking**, tính Dense Vector với `BAAI/bge-m3` và tạo Sparse Index (BM25Okapi), sau đó **LƯU TOÀN BỘ XUỐNG ĐĨA VĨNH VIỄN** (`./rag_storage/`).
>
> ⚠️ **LƯU Ý**: Notebook này **CHỈ CẦN CHẠY ĐÚNG 1 LẦN DUY NHẤT** khi thêm tài liệu mới! Sau khi chạy xong, bạn chỉ việc mở các Notebook Query để hỏi đáp tức thì trong 1 giây mà không cần tính lại vector!


In [ ]:
# 1. CÀI ĐẶT THƯ VIỆN LẬP CHỈ MỤC
!pip install -q -U langchain langchain-community sentence-transformers rank-bm25 pypdf chromadb


In [ ]:
import os
import glob
import pickle
import torch
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from rank_bm25 import BM25Okapi

# Kiểm tra phần cứng
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔥 Thiết bị tính toán: {device.upper()} | GPU: {torch.cuda.get_device_name(0) if device=='cuda' else 'CPU'}")

# 2. Quét tìm toàn bộ tài liệu PDF
pdf_files = list(set(glob.glob("*.pdf") + glob.glob("/kaggle/input/**/*.pdf", recursive=True)))
print(f"📚 Tìm thấy {len(pdf_files)} tài liệu PDF:", pdf_files)

raw_docs = []
for pdf in pdf_files:
    try:
        loader = PyPDFLoader(pdf)
        docs = loader.load()
        print(f"  - Đã nạp: {pdf} ({len(docs)} trang)")
        raw_docs.extend(docs)
    except Exception as e:
        print(f"  ❌ Lỗi file {pdf}: {e}")

# 3. Small-to-Big Chunking
print("\n⏳ Đang phân đoạn tài liệu theo cơ chế Small-to-Big (Parent-Child)...")
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=150, separators=["\n\n", "\n", " ", ""])
parent_docs = parent_splitter.split_documents(raw_docs)

child_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50, separators=["\n\n", "\n", " ", ""])
child_docs = []
for p_idx, p_doc in enumerate(parent_docs):
    p_text = p_doc.page_content
    c_subs = child_splitter.create_documents(
        texts=[p_text],
        metadatas=[{**p_doc.metadata, "parent_id": p_idx, "parent_content": p_text}]
    )
    child_docs.extend(c_subs)

print(f"✅ Hoàn tất cắt chunk: {len(parent_docs)} Parent Chunks | {len(child_docs)} Child Chunks")

# 4. Dense Vector Embedding Model
print("\n⏳ Đang nạp mô hình Dense Embedding BAAI/bge-m3 lên GPU...")
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}
)

# 5. Lưu vĩnh viễn xuống ổ đĩa (Persistent Storage)
PERSIST_DIR = "./rag_storage"
os.makedirs(PERSIST_DIR, exist_ok=True)

print(f"⏳ Đang lập chỉ mục và ghi vĩnh viễn vào thư mục '{PERSIST_DIR}'...")
# A. Lưu ChromaDB xuống đĩa
vector_store = Chroma.from_documents(
    documents=child_docs,
    embedding=embedding_model,
    persist_directory=PERSIST_DIR,
    collection_name="advanced_rag_child_index"
)

# B. Lưu BM25 Sparse Index và metadata xuống file pickle
child_corpus = [doc.page_content.lower().split() for doc in child_docs]
bm25 = BM25Okapi(child_corpus)

with open(f"{PERSIST_DIR}/bm25_index.pkl", "wb") as f:
    pickle.dump({"bm25": bm25, "child_docs": child_docs}, f)

print(f"\n🎉 CHÚC MỪNG! Toàn bộ cơ sở dữ liệu đã được lưu vĩnh viễn tại: {os.path.abspath(PERSIST_DIR)}")
print("👉 Bây giờ bạn có thể mở '01_Advanced_RAG_Inference.ipynb' hoặc '02_Modular_RAG_Inference.ipynb' để sử dụng ngay lập tức mà không cần chạy lại bước này!")
